In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.animation as animation
import matplotlib.font_manager as fm
import networkx as nx
import os

# ── 한글 폰트 설정 (Windows 맑은 고딕 직접 지정) ─────────────
font_path = r"C:\Windows\Fonts\malgun.ttf"
font_prop = fm.FontProperties(fname=font_path)
matplotlib.rc("font", family=font_prop.get_name())
matplotlib.rcParams["axes.unicode_minus"] = False

# ── 그래프 정의 ──────────────────────────────────────────────
G = nx.Graph()
nodes = ["이화만두", "용산토리", "멜트", "류창희국수", "팔도실비집"]
edges = [
    ("이화만두", "용산토리",   26),
    ("이화만두", "멜트",       25),
    ("이화만두", "류창희국수", 30),
    ("멜트",     "류창희국수", 32),
    ("용산토리", "팔도실비집", 41),
]
G.add_nodes_from(nodes)
for u, v, w in edges:
    G.add_edge(u, v, weight=w)

# 원본 이미지와 동일한 위치 배치
pos = {
    "이화만두":   (0.35, 0.80),
    "용산토리":   (0.80, 0.72),
    "멜트":       (0.10, 0.35),
    "류창희국수": (0.45, 0.28),
    "팔도실비집": (0.80, 0.22),
}

# ── DFS 경로 기록 ─────────────────────────────────────────────
def dfs_steps(graph, start):
    visited = []
    edge_steps = []
    stack = [(start, None)]
    visited_set = set()

    while stack:
        node, parent = stack.pop()
        if node in visited_set:
            continue
        visited_set.add(node)
        visited.append(node)
        edge_steps.append((parent, node) if parent else None)

        for nb in reversed(sorted(graph.neighbors(node))):
            if nb not in visited_set:
                stack.append((nb, node))

    return visited, edge_steps

start_node = "이화만두"
visit_order, edge_order = dfs_steps(G, start_node)
n_steps = len(visit_order)

# ── 색상 팔레트 ───────────────────────────────────────────────
COLOR_DEFAULT      = "#F0EAD6"
COLOR_CURRENT      = "#FF6B35"
COLOR_VISITED      = "#4CAF82"
COLOR_EDGE_DEFAULT = "#BDBDBD"
COLOR_EDGE_ACTIVE  = "#FF6B35"
COLOR_EDGE_DONE    = "#4CAF82"
COLOR_BG           = "#FAFAF8"

# ── Figure 설정 ───────────────────────────────────────────────
fig, (ax_graph, ax_log) = plt.subplots(
    1, 2, figsize=(13, 7),
    gridspec_kw={"width_ratios": [2, 1]},
    facecolor=COLOR_BG
)
fig.suptitle("맛집 그래프 DFS 탐색", fontsize=18, fontweight="bold",
             color="#2C2C2C", y=0.97)

def draw_frame(step):
    ax_graph.cla()
    ax_log.cla()
    ax_graph.set_facecolor(COLOR_BG)
    ax_log.set_facecolor(COLOR_BG)
    ax_graph.axis("off")
    ax_log.axis("off")
    ax_graph.set_xlim(-0.05, 1.05)
    ax_graph.set_ylim(-0.05, 1.05)

    visited_so_far = set(visit_order[:step + 1])
    current_node   = visit_order[step]
    active_edge    = edge_order[step]
    done_edges     = {edge_order[i] for i in range(1, step + 1)
                      if edge_order[i] is not None}

    node_colors = []
    for n in G.nodes():
        if n == current_node:
            node_colors.append(COLOR_CURRENT)
        elif n in visited_so_far:
            node_colors.append(COLOR_VISITED)
        else:
            node_colors.append(COLOR_DEFAULT)

    edge_colors, edge_widths = [], []
    for u, v in G.edges():
        is_active = active_edge and (active_edge == (u,v) or active_edge == (v,u))
        is_done   = (u,v) in done_edges or (v,u) in done_edges
        if is_active:
            edge_colors.append(COLOR_EDGE_ACTIVE); edge_widths.append(4.5)
        elif is_done:
            edge_colors.append(COLOR_EDGE_DONE);   edge_widths.append(3.0)
        else:
            edge_colors.append(COLOR_EDGE_DEFAULT); edge_widths.append(1.5)

    nx.draw_networkx_nodes(G, pos, ax=ax_graph,
                           node_color=node_colors, node_size=3000,
                           edgecolors="#555555", linewidths=2)
    nx.draw_networkx_labels(G, pos, ax=ax_graph,
                            font_size=11, font_weight="bold", font_color="#2C2C2C",
                            font_family=font_prop.get_name())
    nx.draw_networkx_edges(G, pos, ax=ax_graph,
                           edge_color=edge_colors, width=edge_widths)
    nx.draw_networkx_edge_labels(
        G, pos, ax=ax_graph,
        edge_labels={(u, v): f"{w}분" for u, v, w in edges},
        font_size=9, font_color="#555555",
        font_family=font_prop.get_name(),
        bbox=dict(boxstyle="round,pad=0.2", fc=COLOR_BG, ec="none", alpha=0.8)
    )

    ax_graph.set_title(
        f"Step {step+1} / {n_steps}  →  현재 방문: {current_node}",
        fontsize=13, color="#2C2C2C", pad=10
    )

    ax_log.set_xlim(0, 1); ax_log.set_ylim(0, 1)
    ax_log.text(0.5, 0.96, "방문 기록", ha="center", va="top",
                fontsize=13, fontweight="bold", color="#2C2C2C")
    ax_log.axhline(y=0.91, color="#CCCCCC", linewidth=1, xmin=0.05, xmax=0.95)

    for i, node in enumerate(visit_order[:step + 1]):
        y = 0.86 - i * 0.09
        color = COLOR_CURRENT if i == step else COLOR_VISITED
        ax_log.text(0.12, y, f"{i+1}.", fontsize=11, color="#888888", va="center")
        ax_log.text(0.22, y, node,      fontsize=11, color=color,
                    fontweight="bold", va="center")

    patches = [
        mpatches.Patch(color=COLOR_DEFAULT, label="미방문"),
        mpatches.Patch(color=COLOR_CURRENT, label="현재 방문"),
        mpatches.Patch(color=COLOR_VISITED, label="방문 완료"),
    ]
    ax_graph.legend(handles=patches, loc="lower left", fontsize=9,
                    framealpha=0.85, facecolor=COLOR_BG, edgecolor="#CCCCCC")

ani = animation.FuncAnimation(
    fig, draw_frame,
    frames=list(range(n_steps)),
    interval=1200,
    repeat=True,
    repeat_delay=2000,
)

# 저장 경로: 이 스크립트와 같은 폴더에 저장
output_path = os.path.join(
    os.path.dirname(os.path.abspath(__file__)),
    "dfs_restaurants.gif"
)

writer = animation.PillowWriter(fps=1)
ani.save(output_path, writer=writer, dpi=120)
print(f"저장 완료: {output_path}")
print(f"DFS 방문 순서: {' → '.join(visit_order)}")


In [ ]:
import os
import platform
import matplotlib.pyplot as plt
import matplotlib.animation as animation

# 1. 한글 폰트 설정 (OS별 상이)
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# 2. 그래프 데이터 및 노드 좌표 정의
positions = {
    '이화만두': (3.0, 3.8),
    '멜트': (1.2, 1.5),
    '류창희국수': (3.5, 0.8),
    '용산토리': (5.5, 3.2),
    '팔도실비집': (6.2, 1.2)
}

edges = [
    ('이화만두', '멜트', '25분'),
    ('이화만두', '류창희국수', '30분'),
    ('이화만두', '용산토리', '26분'),
    ('멜트', '류창희국수', '32분'),
    ('용산토리', '팔도실비집', '41분')
]

# 3. BFS 탐색 단계별 상태 정의 (이화만두 -> 류창희국수 -> 멜트 -> 용산토리 -> 팔도실비집 순)
steps = [
    {
        'curr': '이화만두',
        'visited_done': [],
        'history': ['이화만두'],
        'edges_green': []
    },
    {
        'curr': '류창희국수',
        'visited_done': ['이화만두'],
        'history': ['이화만두', '류창희국수'],
        'edges_green': [('이화만두', '류창희국수')]
    },
    {
        'curr': '멜트',
        'visited_done': ['이화만두', '류창희국수'],
        'history': ['이화만두', '류창희국수', '멜트'],
        'edges_green': [('이화만두', '류창희국수'), ('이화만두', '멜트')]
    },
    {
        'curr': '용산토리',
        'visited_done': ['이화만두', '류창희국수', '멜트'],
        'history': ['이화만두', '류창희국수', '멜트', '용산토리'],
        'edges_green': [('이화만두', '류창희국수'), ('이화만두', '멜트'), ('이화만두', '용산토리')]
    },
    {
        'curr': '팔도실비집',
        'visited_done': ['이화만두', '류창희국수', '멜트', '용산토리'],
        'history': ['이화만두', '류창희국수', '멜트', '용산토리', '팔도실비집'],
        'edges_green': [('이화만두', '류창희국수'), ('이화만두', '멜트'), ('이화만두', '용산토리'), ('용산토리', '팔도실비집')]
    }
]

# 4. 시각화 피겨 및 축 설정
fig, ax = plt.subplots(figsize=(11, 6.5))
fig.patch.set_facecolor('#F9F9F7')

def update(frame):
    ax.clear()
    ax.set_facecolor('#F9F9F7')
    ax.set_xlim(0, 9)
    ax.set_ylim(0, 5)
    ax.axis('off')

    step = steps[frame]
    curr_node = step['curr']
    visited_done = step['visited_done']
    history = step['history']
    edges_green = step['edges_green']

    # 타이틀 및 진행 상태 표시
    ax.text(3.5, 4.6, "맛집 그래프 BFS 탐색", fontsize=16, fontweight='bold', ha='center', color='#2C3E50')
    ax.text(0.8, 4.3, f"Step {frame + 1} / 5   →   현재 방문: {curr_node}", fontsize=11, color='#555555')

    # 간선(Edge) 그리기
    for u, v, weight in edges:
        x1, y1 = positions[u]
        x2, y2 = positions[v]

        is_green = (u, v) in edges_green or (v, u) in edges_green
        edge_color = '#2ECC71' if is_green else '#B0B0B0'
        edge_width = 3.5 if is_green else 1.5

        ax.plot([x1, x2], [y1, y2], color=edge_color, linewidth=edge_width, zorder=1)

        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        ax.text(mx, my, weight, ha='center', va='center', fontsize=9, color='#333333',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='#F9F9F7', edgecolor='none', alpha=0.9), zorder=2)

    # 노드(Node) 그리기
    for node, (x, y) in positions.items():
        if node == curr_node:
            facecolor = '#E74C3C'
            textcolor = 'white'
        elif node in visited_done:
            facecolor = '#2ECC71'
            textcolor = 'white'
        else:
            facecolor = '#F5F2EB'
            textcolor = '#333333'

        circle = plt.Circle((x, y), radius=0.42, facecolor=facecolor, edgecolor='#333333', linewidth=1.2, zorder=3)
        ax.add_patch(circle)

        display_name = node[:2] + '\n' + node[2:] if len(node) > 2 else node
        ax.text(x, y, display_name, ha='center', va='center', fontsize=10, fontweight='bold', color=textcolor, zorder=4)

    # 우측 상단 패널: 방문 기록
    ax.text(7.5, 4.3, "방문 기록", fontsize=13, fontweight='bold', color='#333333')
    ax.plot([7.5, 8.5], [4.15, 4.15], color='#D0D0D0', linewidth=1)

    for idx, name in enumerate(history):
        y_pos = 3.8 - idx * 0.35
        item_color = '#E74C3C' if name == curr_node else '#2ECC71'
        ax.text(7.5, y_pos, f"{idx + 1}.  {name}", fontsize=11, color=item_color, fontweight='bold')

    # 좌측 하단 패널: 범례
    legends = [('미방문', '#F5F2EB'), ('현재 방문', '#E74C3C'), ('방문 완료', '#2ECC71')]
    lx, ly = 0.5, 0.4
    for label, color in legends:
        rect = plt.Rectangle((lx, ly), 0.22, 0.12, facecolor=color, edgecolor='#555555', linewidth=0.8)
        ax.add_patch(rect)
        ax.text(lx + 0.32, ly + 0.04, label, fontsize=9, va='center', color='#555555')
        ly += 0.22

# 5. 애니메이션 객체 생성 및 GIF 저장 (Pillow 사용)
# interval 매개변수 대신 fps(초당 프레임)로 전환 속도를 조절합니다. fps=1은 1초에 한 프레임씩 보여줍니다.
anim = animation.FuncAnimation(fig, update, frames=len(steps), repeat=False)
output_filename = 'restaurant_bfs_exploration.gif'

# FFmpeg 대신 파이썬 내장/Pillow 라이브러리를 사용하여 저장
anim.save(output_filename, writer='pillow', fps=1, dpi=120)

plt.close()
print(f"성공적으로 GIF 파일이 저장되었습니다: {os.path.abspath(output_filename)}")

In [ ]:
import os
import platform
import matplotlib.pyplot as plt
import matplotlib.animation as animation

# 1. 한글 폰트 설정 (OS별 상이)
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# 2. 그래프 데이터 및 노드 좌표 정의
positions = {
    '이화만두': (3.0, 3.8),
    '멜트': (1.2, 1.5),
    '류창희국수': (3.5, 0.8),
    '용산토리': (5.5, 3.2),
    '팔도실비집': (6.2, 1.2)
}

# (노드1, 노드2, 가중치 텍스트, 가중치 정수값)
edges = [
    ('이화만두', '멜트', '25분', 25),
    ('이화만두', '류창희국수', '30분', 30),
    ('이화만두', '용산토리', '26분', 26),
    ('멜트', '류창희국수', '32분', 32),
    ('용산토리', '팔도실비집', '41분', 41)
]

# 3. 프림 알고리즘 탐색 단계별 상태 정의
# 시작 노드: 이화만두
# Step 1: 이화만두 -> 멜트 (25분, 최소)
# Step 2: 이화만두 -> 용산토리 (26분, 남은 것 중 최소)
# Step 3: 이화만두 -> 류창희국수 (30분, 남은 것 중 최소)
# Step 4: 용산토리 -> 팔도실비집 (41분)
steps = [
    {
        'curr': '이화만두',
        'visited_done': [],
        'history': ['이화만두'],
        'edges_green': []
    },
    {
        'curr': '멜트',
        'visited_done': ['이화만두'],
        'history': ['이화만두', '멜트'],
        'edges_green': [('이화만두', '멜트')]
    },
    {
        'curr': '용산토리',
        'visited_done': ['이화만두', '멜트'],
        'history': ['이화만두', '멜트', '용산토리'],
        'edges_green': [('이화만두', '멜트'), ('이화만두', '용산토리')]
    },
    {
        'curr': '류창희국수',
        'visited_done': ['이화만두', '멜트', '용산토리'],
        'history': ['이화만두', '멜트', '용산토리', '류창희국수'],
        'edges_green': [('이화만두', '멜트'), ('이화만두', '용산토리'), ('이화만두', '류창희국수')]
    },
    {
        'curr': '팔도실비집',
        'visited_done': ['이화만두', '멜트', '용산토리', '류창희국수'],
        'history': ['이화만두', '멜트', '용산토리', '류창희국수', '팔도실비집'],
        'edges_green': [('이화만두', '멜트'), ('이화만두', '용산토리'), ('이화만두', '류창희국수'), ('용산토리', '팔도실비집')]
    }
]

# 4. 시각화 피겨 및 축 설정
fig, ax = plt.subplots(figsize=(11, 6.5))
fig.patch.set_facecolor('#F9F9F7')


def update(frame):
    ax.clear()
    ax.set_facecolor('#F9F9F7')
    ax.set_xlim(0, 9)
    ax.set_ylim(0, 5)
    ax.axis('off')

    step = steps[frame]
    curr_node = step['curr']
    visited_done = step['visited_done']
    history = step['history']
    edges_green = step['edges_green']

    # 타이틀 및 진행 상태 표시
    ax.text(3.5, 4.6, "맛집 그래프 프림(Prim) 알고리즘 탐색", fontsize=16, fontweight='bold', ha='center', color='#2C3E50')
    ax.text(0.8, 4.3, f"Step {frame + 1} / 5   →   현재 방문: {curr_node}", fontsize=11, color='#555555')

    # 간선(Edge) 그리기
    for u, v, weight_str, weight_int in edges:
        x1, y1 = positions[u]
        x2, y2 = positions[v]

        is_green = (u, v) in edges_green or (v, u) in edges_green
        edge_color = '#2ECC71' if is_green else '#B0B0B0'
        edge_width = 3.5 if is_green else 1.5

        ax.plot([x1, x2], [y1, y2], color=edge_color, linewidth=edge_width, zorder=1)

        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        ax.text(mx, my, weight_str, ha='center', va='center', fontsize=9, color='#333333',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='#F9F9F7', edgecolor='none', alpha=0.9), zorder=2)

    # 노드(Node) 그리기
    for node, (x, y) in positions.items():
        if node == curr_node:
            facecolor = '#E74C3C'
            textcolor = 'white'
        elif node in visited_done:
            facecolor = '#2ECC71'
            textcolor = 'white'
        else:
            facecolor = '#F5F2EB'
            textcolor = '#333333'

        circle = plt.Circle((x, y), radius=0.42, facecolor=facecolor, edgecolor='#333333', linewidth=1.2, zorder=3)
        ax.add_patch(circle)

        display_name = node[:2] + '\n' + node[2:] if len(node) > 2 else node
        ax.text(x, y, display_name, ha='center', va='center', fontsize=10, fontweight='bold', color=textcolor, zorder=4)

    # 우측 상단 패널: 방문 기록 (MST 편입 순서)
    ax.text(7.5, 4.3, "방문 기록(MST 추가)", fontsize=12, fontweight='bold', color='#333333')
    ax.plot([7.5, 8.8], [4.15, 4.15], color='#D0D0D0', linewidth=1)

    for idx, name in enumerate(history):
        y_pos = 3.8 - idx * 0.35
        item_color = '#E74C3C' if name == curr_node else '#2ECC71'
        ax.text(7.5, y_pos, f"{idx + 1}.  {name}", fontsize=11, color=item_color, fontweight='bold')

    # 좌측 하단 패널: 범례
    legends = [('미방문', '#F5F2EB'), ('현재 방문', '#E74C3C'), ('방문 완료', '#2ECC71')]
    lx, ly = 0.5, 0.4
    for label, color in legends:
        rect = plt.Rectangle((lx, ly), 0.22, 0.12, facecolor=color, edgecolor='#555555', linewidth=0.8)
        ax.add_patch(rect)
        ax.text(lx + 0.32, ly + 0.04, label, fontsize=9, va='center', color='#555555')
        ly += 0.22


# 5. 애니메이션 객체 생성 및 GIF 저장
anim = animation.FuncAnimation(fig, update, frames=len(steps), repeat=False)
output_filename = 'restaurant_prim_exploration.gif'

anim.save(output_filename, writer='pillow', fps=1, dpi=120)

plt.close()
print(f"성공적으로 GIF 파일이 저장되었습니다: {os.path.abspath(output_filename)}")

In [ ]:
import os
import platform
import matplotlib.pyplot as plt
import matplotlib.animation as animation

# 1. 한글 폰트 설정 (OS별 상이)
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# 2. 그래프 데이터 및 노드 좌표 정의 (이미지 레이아웃 기준)
positions = {
    '이화만두': (3.0, 3.8),
    '멜트': (1.2, 1.5),
    '류창희국수': (3.5, 0.8),
    '용산토리': (5.5, 3.2),
    '팔도실비집': (6.2, 1.2)
}

# 모든 간선 리스트
edges = [
    ('이화만두', '멜트', '25분'),
    ('이화만두', '류창희국수', '30분'),
    ('이화만두', '용산토리', '26분'),
    ('멜트', '류창희국수', '32분'),
    ('용산토리', '팔도실비집', '41분')
]

# 3. 크루스칼 알고리즘 단계별 상태 정의 (간선 가중치 오름차순 정렬 기준)
# 1) 이화만두 - 멜트 (25분) -> 선택
# 2) 이화만두 - 용산토리 (26분) -> 선택
# 3) 이화만두 - 류창희국수 (30분) -> 선택
# 4) 멜트 - 류창희국수 (32분) -> 사이클 발생으로 제외(Skip)
# 5) 용산토리 - 팔도실비집 (41분) -> 선택
steps = [
    {
        'msg': "최단 간선 '이화만두 - 멜트(25분)' 선택",
        'edges_green': [('이화만두', '멜트')],
        'edge_red': None,
        'nodes_green': ['이화만두', '멜트'],
        'history': [('1. 이화만두-멜트 (25분)', '#2ECC71')]
    },
    {
        'msg': "다음 최단 간선 '이화만두 - 용산토리(26분)' 선택",
        'edges_green': [('이화만두', '멜트'), ('이화만두', '용산토리')],
        'edge_red': None,
        'nodes_green': ['이화만두', '멜트', '용산토리'],
        'history': [('1. 이화만두-멜트 (25분)', '#2ECC71'), ('2. 이화만두-용산토리 (26분)', '#2ECC71')]
    },
    {
        'msg': "다음 최단 간선 '이화만두 - 류창희국수(30분)' 선택",
        'edges_green': [('이화만두', '멜트'), ('이화만두', '용산토리'), ('이화만두', '류창희국수')],
        'nodes_green': ['이화만두', '멜트', '용산토리', '류창희국수'],
        'edge_red': None,
        'history': [('1. 이화만두-멜트 (25분)', '#2ECC71'), ('2. 이화만두-용산토리 (26분)', '#2ECC71'),
                    ('3. 이화만두-류창희국수 (30분)', '#2ECC71')]
    },
    {
        'msg': "'멜트 - 류창희국수(32분)' 검토 → 사이클 발생으로 제외!",
        'edges_green': [('이화만두', '멜트'), ('이화만두', '용산토리'), ('이화만두', '류창희국수')],
        'nodes_green': ['이화만두', '멜트', '용산토리', '류창희국수'],
        'edge_red': ('멜트', '류창희국수'),  # 사이클 간선 빨간색 표시
        'history': [('1. 이화만두-멜트 (25분)', '#2ECC71'), ('2. 이화만두-용산토리 (26분)', '#2ECC71'),
                    ('3. 이화만두-류창희국수 (30분)', '#2ECC71'), ('4. 멜트-류창희국수 (32분 제외)', '#E74C3C')]
    },
    {
        'msg': "마지막 간선 '용산토리 - 팔도실비집(41분)' 선택 → MST 완성",
        'edges_green': [('이화만두', '멜트'), ('이화만두', '용산토리'), ('이화만두', '류창희국수'), ('용산토리', '팔도실비집')],
        'nodes_green': ['이화만두', '멜트', '용산토리', '류창희국수', '팔도실비집'],
        'edge_red': None,
        'history': [('1. 이화만두-멜트 (25분)', '#2ECC71'), ('2. 이화만두-용산토리 (26분)', '#2ECC71'),
                    ('3. 이화만두-류창희국수 (30분)', '#2ECC71'), ('4. 멜트-류창희국수 (제외)', '#B0B0B0'),
                    ('5. 용산토리-팔도실비집 (41분)', '#2ECC71')]
    }
]

# 4. 시각화 피겨 및 축 설정
fig, ax = plt.subplots(figsize=(11, 6.5))
fig.patch.set_facecolor('#F9F9F7')


def update(frame):
    ax.clear()
    ax.set_facecolor('#F9F9F7')
    ax.set_xlim(0, 9.5)
    ax.set_ylim(0, 5)
    ax.axis('off')

    step = steps[frame]
    edges_green = step['edges_green']
    edge_red = step['edge_red']
    nodes_green = step['nodes_green']
    history = step['history']

    # 타이틀 및 진행 상태 표시
    ax.text(3.5, 4.6, "맛집 그래프 크루스칼(Kruskal) 알고리즘", fontsize=16, fontweight='bold', ha='center', color='#2C3E50')
    ax.text(0.8, 4.3, f"Step {frame + 1} / 5   →   {step['msg']}", fontsize=11, color='#555555')

    # 간선(Edge) 그리기
    for u, v, weight in edges:
        x1, y1 = positions[u]
        x2, y2 = positions[v]

        is_green = (u, v) in edges_green or (v, u) in edges_green
        is_red = edge_red is not None and (
                    (u == edge_red[0] and v == edge_red[1]) or (v == edge_red[0] and u == edge_red[1]))

        if is_red:
            edge_color = '#E74C3C'  # 제외된 사이클 간선 (빨간색 점선)
            edge_width = 3.5
            linestyle = '--'
        elif is_green:
            edge_color = '#2ECC71'  # 선택된 트리 간선 (초록색 실선)
            edge_width = 3.5
            linestyle = '-'
        else:
            edge_color = '#B0B0B0'  # 대기 상태 간선 (회색 실선)
            edge_width = 1.5
            linestyle = '-'

        ax.plot([x1, x2], [y1, y2], color=edge_color, linewidth=edge_width, linestyle=linestyle, zorder=1)

        # 간선 가중치(분) 표시
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        ax.text(mx, my, weight, ha='center', va='center', fontsize=9, color='#333333',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='#F9F9F7', edgecolor='none', alpha=0.9), zorder=2)

    # 노드(Node) 그리기
    for node, (x, y) in positions.items():
        if node in nodes_green:
            facecolor = '#2ECC71'  # MST에 포함된 노드 그룹 (초록색)
            textcolor = 'white'
        else:
            facecolor = '#F5F2EB'  # 아직 연결되지 않은 노드 (연베이지색)
            textcolor = '#333333'

        circle = plt.Circle((x, y), radius=0.42, facecolor=facecolor, edgecolor='#333333', linewidth=1.2, zorder=3)
        ax.add_patch(circle)

        display_name = node[:2] + '\n' + node[2:] if len(node) > 2 else node
        ax.text(x, y, display_name, ha='center', va='center', fontsize=10, fontweight='bold', color=textcolor, zorder=4)

    # 우측 상단 패널: 간선 선택 기록
    ax.text(7.2, 4.3, "간선 정렬 및 선택 기록", fontsize=12, fontweight='bold', color='#333333')
    ax.plot([7.2, 9.2], [4.15, 4.15], color='#D0D0D0', linewidth=1)

    for idx, (text, color) in enumerate(history):
        y_pos = 3.8 - idx * 0.35
        ax.text(7.2, y_pos, text, fontsize=10, color=color, fontweight='bold')

    # 좌측 하단 패널: 범례
    legends = [('대기 간선', '#F5F2EB'), ('선택 간선', '#2ECC71'), ('사이클(제외)', '#E74C3C')]
    lx, ly = 0.5, 0.4
    for label, color in legends:
        rect = plt.Rectangle((lx, ly), 0.22, 0.12, facecolor=color, edgecolor='#555555', linewidth=0.8)
        ax.add_patch(rect)
        ax.text(lx + 0.32, ly + 0.04, label, fontsize=9, va='center', color='#555555')
        ly += 0.22


# 5. 애니메이션 객체 생성 및 GIF 저장 (Pillow 사용으로 FFmpeg 불필요)
anim = animation.FuncAnimation(fig, update, frames=len(steps), repeat=False)
output_filename = 'restaurant_kruskal_exploration.gif'

anim.save(output_filename, writer='pillow', fps=1, dpi=120)

plt.close()
print(f"성공적으로 크루스칼 GIF 파일이 저장되었습니다: {os.path.abspath(output_filename)}")